In [ ]:
import pandas as pd
import numpy as np

In [ ]:
prev = pd.read_csv("D:/AI/HomeCredit/Dataset/previous_application.csv")
prev.head()

In [ ]:
prev = prev.replace(365243,np.nan)
prev.head()

In [ ]:
prev['APP_CREDIT_RATIO'] = prev['AMT_APPLICATION'] / prev['AMT_CREDIT']
prev['DOWN_PAYMENT_RATIO'] = prev['AMT_DOWN_PAYMENT'] / prev['AMT_CREDIT']
prev['CREDIT_DURATION'] = prev['DAYS_LAST_DUE'] - prev['DAYS_FIRST_DUE']
prev['DAYS_DECISION_ABS'] = abs(prev['DAYS_DECISION'])

In [ ]:
prev = pd.get_dummies(prev)
prev.shape

In [ ]:
list(prev.columns)

In [ ]:
approved = prev[prev['NAME_CONTRACT_STATUS_Approved'] == 1]
rejected = prev[prev['NAME_CONTRACT_STATUS_Refused'] == 1]

In [ ]:
prev_agg = prev.groupby('SK_ID_CURR').agg({
    'AMT_CREDIT': ['mean', 'max'],
    'AMT_APPLICATION': ['mean'],
    'APP_CREDIT_RATIO': ['mean'],
    'DOWN_PAYMENT_RATIO': ['mean'],
    'CNT_PAYMENT': ['mean'],
    'CREDIT_DURATION': ['mean'],
    'DAYS_DECISION': ['mean', 'max'],
})

In [ ]:
approved_agg = approved.groupby('SK_ID_CURR').agg({
    'AMT_CREDIT': ['mean', 'max'],
    'APP_CREDIT_RATIO': ['mean'],
})

In [ ]:
rejected_agg = rejected.groupby('SK_ID_CURR').agg({
    'AMT_APPLICATION': ['mean'],
})

In [ ]:
prev_counts = prev.groupby('SK_ID_CURR').agg({
    'SK_ID_PREV': 'count'
}).rename(columns={'SK_ID_PREV': 'PREV_APP_COUNT'})

In [ ]:
prev_agg.head()

In [ ]:
prev_agg.columns = [
    '_'.join(col).upper() if isinstance(col, tuple) else col
    for col in prev_agg.columns
]

prev_agg.reset_index(inplace=True)

prev_final = prev_agg.copy()

In [ ]:
prev_final.head()

In [ ]:
approved_agg.columns = [
    '_'.join(col).upper() if isinstance(col, tuple) else col
    for col in approved_agg.columns
]
approved_agg.reset_index(inplace=True)

In [ ]:
rejected_agg.columns = [
    '_'.join(col).upper() if isinstance(col, tuple) else col
    for col in rejected_agg.columns
]
rejected_agg.reset_index(inplace=True)

In [ ]:
prev_final = prev_final.merge(approved_agg, on='SK_ID_CURR', how='left')
prev_final = prev_final.merge(rejected_agg, on='SK_ID_CURR', how='left')
prev_final = prev_final.merge(prev_counts, on='SK_ID_CURR', how='left')

In [ ]:
prev_final.head()

In [ ]:
prev_final.to_csv("D:/AI/HomeCredit/Processed_data/prev_final.csv", index=False)